In [154]:
print("hEy")

hEy


In [155]:
import pandas as pd
import numpy as np
import json

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.linear_model import LinearRegression
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor, StackingRegressor
from sklearn.svm import SVR
from sklearn.pipeline import make_pipeline
from sklearn.metrics import mean_squared_error, r2_score

# =========================
# LOAD DATA
# =========================
matches = pd.read_csv("matches.csv")
deliveries = pd.read_csv("deliveries.csv")

print("Initial shapes:", matches.shape, deliveries.shape)


Initial shapes: (1095, 20) (260920, 17)


In [156]:

# =========================
# PREPROCESSING
# =========================

# P1
matches = matches.dropna(subset=['winner'])

# P2, P3
matches['city'] = matches['city'].fillna('Unknown')
matches['method'] = matches['method'].fillna('Normal')

# P4
matches['date'] = pd.to_datetime(matches['date'])
matches['match_year'] = matches['date'].dt.year
matches['match_month'] = matches['date'].dt.month

# P5 - TEAM NORMALIZATION
team_map = {
    'Delhi Daredevils': 'Delhi Capitals',
    'Kings XI Punjab': 'Punjab Kings',
    'Rising Pune Supergiant': 'Rising Pune Supergiants',
    'Royal Challengers Bangalore': 'Royal Challengers Bengaluru'
}

for col in ['team1','team2','toss_winner','winner']:
    matches[col] = matches[col].replace(team_map)

deliveries['batting_team'] = deliveries['batting_team'].replace(team_map)
deliveries['bowling_team'] = deliveries['bowling_team'].replace(team_map)

# P6
deliveries = deliveries[deliveries['inning'].isin([1,2])]

# P7, P8
deliveries['extras_type'] = deliveries['extras_type'].fillna('normal')
deliveries[['player_dismissed','dismissal_kind']] = deliveries[['player_dismissed','dismissal_kind']].fillna('none')


In [157]:

# =========================
# FEATURE ENGINEERING
# =========================

# merge
df = deliveries.merge(matches, left_on='match_id', right_on='id')

# TARGET (final score per innings)
final_score = df.groupby(['match_id','inning'])['total_runs'].sum().reset_index()
final_score.rename(columns={'total_runs':'final_score'}, inplace=True)

# FIRST 10 OVERS
df_10 = df[df['over'] < 10]

# AGG FEATURES
agg = df_10.groupby(['match_id','inning']).agg({
    'total_runs':'sum',
    'is_wicket':'sum',
    'batsman_runs':lambda x: ((x==4)|(x==6)).sum(),
    'extra_runs':'sum'
}).reset_index()

agg.columns = ['match_id','inning','runs_first10','wickets_first10','boundaries_first10','extras_first10']

# DOT BALLS
df_10['is_dot'] = (df_10['total_runs']==0).astype(int)
dots = df_10.groupby(['match_id','inning'])['is_dot'].sum().reset_index()
dots.rename(columns={'is_dot':'dots_first10'}, inplace=True)

# POWERPLAY
pp = df_10[df_10['over'] < 6].groupby(['match_id','inning']).agg({
    'total_runs':'sum',
    'is_wicket':'sum'
}).reset_index()
pp.columns = ['match_id','inning','powerplay_runs','powerplay_wickets']

# BATTERS USED
batters = df_10.groupby(['match_id','inning'])['batter'].nunique().reset_index()
batters.rename(columns={'batter':'batters_used_10'}, inplace=True)

# LEGAL BALLS
legal = df_10[~df_10['extras_type'].isin(['wides','noballs'])] \
    .groupby(['match_id','inning']).size().reset_index(name='legal_balls_10')

# MERGE ALL
features = final_score.merge(agg, on=['match_id','inning'])
features = features.merge(dots, on=['match_id','inning'])
features = features.merge(pp, on=['match_id','inning'])
features = features.merge(batters, on=['match_id','inning'])
features = features.merge(legal, on=['match_id','inning'])


C:\Users\shukl\AppData\Local\Temp\ipykernel_8304\1466221683.py:26: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_10['is_dot'] = (df_10['total_runs']==0).astype(int)


In [158]:

# =========================
# DERIVED FEATURES
# =========================

features['run_rate_10'] = features['runs_first10'] / (features['legal_balls_10']/6)
features['boundary_rate_10'] = features['boundaries_first10'] / (features['legal_balls_10']/6)
features['dot_rate_10'] = features['dots_first10'] / features['legal_balls_10']
features['pp_rr'] = features['powerplay_runs'] / 6


In [159]:

# =========================
# ADD MATCH CONTEXT
# =========================

match_info = matches[['id','venue','city','toss_winner','toss_decision',
                      'match_year','match_month','match_type']]

data = features.merge(match_info, left_on='match_id', right_on='id')


In [160]:

# =========================
# ENCODING
# =========================

encoders = {}
cat_cols = ['venue','city','toss_winner','toss_decision','match_type']

for col in cat_cols:
    le = LabelEncoder()
    data[col] = le.fit_transform(data[col])
    encoders[col] = le


In [161]:

# =========================
# FINAL DATASET
# =========================

X = data.drop(columns=['final_score','match_id','id'])
y = data['final_score']


In [162]:

# =========================
# TRAIN TEST SPLIT
# =========================

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)


In [163]:

# =========================
# MODELS
# =========================

results = {}

def evaluate(name, model, Xtr, Xte):
    model.fit(Xtr, y_train)
    pred = model.predict(Xte)
    mse = mean_squared_error(y_test, pred)
    rmse = np.sqrt(mse)
    r2 = r2_score(y_test, pred)
    results[name] = {'MSE':mse, 'RMSE':rmse, 'R2':r2}
    return model

# Scaling for LR & SVR
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)


In [164]:
# Models
lr = evaluate("Linear Regression", LinearRegression(), X_train_scaled, X_test_scaled)
print("Model Traind sucessfully\n")

Model Traind sucessfully



In [165]:
dt = evaluate("Decision Tree", DecisionTreeRegressor(max_depth=10, random_state=42), X_train, X_test)
print("Model Traind sucessfully\n")


Model Traind sucessfully



In [166]:
rf = evaluate("Random Forest", RandomForestRegressor(n_estimators=200, max_depth=15, random_state=42, n_jobs=-1), X_train, X_test)
print("Model Traind sucessfully\n")


Model Traind sucessfully



In [167]:
svr = evaluate("SVR", SVR(C=200, gamma=0.05, epsilon=0.1), X_train_scaled, X_test_scaled)
print("Model Traind sucessfully\n")



Model Traind sucessfully



In [168]:

# =========================
# STACKING
# =========================

# select top 2 by RMSE
sorted_models = sorted(results.items(), key=lambda x: x[1]['RMSE'])
best2 = [sorted_models[0][0], sorted_models[1][0]]

model_map = {
    "Linear Regression": make_pipeline(StandardScaler(), LinearRegression()),
    "Random Forest": RandomForestRegressor(n_estimators=200, max_depth=15, random_state=42, n_jobs=-1),
    "Decision Tree": DecisionTreeRegressor(max_depth=10, random_state=42),
    "SVR": make_pipeline(StandardScaler(), SVR(C=200, gamma=0.05, epsilon=0.1))
}

estimators = [(name, model_map[name]) for name in best2]

stack = StackingRegressor(
    estimators=estimators,
    final_estimator=LinearRegression(),
    cv=5,
    passthrough=False
)

stack.fit(X_train, y_train)
pred = stack.predict(X_test)

mse = mean_squared_error(y_test, pred)
rmse = np.sqrt(mse)
r2 = r2_score(y_test, pred)

results["Stacking"] = {'MSE':mse, 'RMSE':rmse, 'R2':r2}



In [169]:
# =========================
# SAVE RESULTS
# =========================

output = {
    "results": results,
    "best2_models": best2,
    "n_train": len(X_train),
    "n_test": len(X_test),
    "features": list(X.columns)
}

with open("model_results.json", "w") as f:
    json.dump(output, f, indent=4)

print("✅ Done! Results saved to model_results.json")
pd.DataFrame(results)

✅ Done! Results saved to model_results.json


,Linear Regression,Decision Tree,Random Forest,SVR,Stacking
MSE,572.485886,750.892296,529.463562,657.743058,519.331279
RMSE,23.926677,27.402414,23.010075,25.646502,22.788841
R2,0.451875,0.281060,0.493067,0.370246,0.502768


In [170]:
import pandas as pd
import numpy as np

# preprocessing
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.impute import KNNImputer
from sklearn.model_selection import train_test_split

# models
from sklearn.linear_model import LinearRegression
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor, AdaBoostRegressor, GradientBoostingRegressor
from sklearn.svm import SVR

# boosting external
from xgboost import XGBRegressor
from lightgbm import LGBMRegressor

# stacking
from sklearn.ensemble import StackingRegressor

# metrics
from sklearn.metrics import mean_squared_error, r2_score

# visualization
import matplotlib.pyplot as plt
import seaborn as sns

In [171]:
matches = pd.read_csv('matches.csv')
deliveries = pd.read_csv('deliveries.csv')

In [172]:
matches.info()


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1095 entries, 0 to 1094
Data columns (total 20 columns):
 #   Column           Non-Null Count  Dtype  
---  ------           --------------  -----  
 0   id               1095 non-null   int64  
 1   season           1095 non-null   object 
 2   city             1044 non-null   object 
 3   date             1095 non-null   object 
 4   match_type       1095 non-null   object 
 5   player_of_match  1090 non-null   object 
 6   venue            1095 non-null   object 
 7   team1            1095 non-null   object 
 8   team2            1095 non-null   object 
 9   toss_winner      1095 non-null   object 
 10  toss_decision    1095 non-null   object 
 11  winner           1090 non-null   object 
 12  result           1095 non-null   object 
 13  result_margin    1076 non-null   float64
 14  target_runs      1092 non-null   float64
 15  target_overs     1092 non-null   float64
 16  super_over       1095 non-null   object 
 17  method        

In [173]:
deliveries.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 260920 entries, 0 to 260919
Data columns (total 17 columns):
 #   Column            Non-Null Count   Dtype 
---  ------            --------------   ----- 
 0   match_id          260920 non-null  int64 
 1   inning            260920 non-null  int64 
 2   batting_team      260920 non-null  object
 3   bowling_team      260920 non-null  object
 4   over              260920 non-null  int64 
 5   ball              260920 non-null  int64 
 6   batter            260920 non-null  object
 7   bowler            260920 non-null  object
 8   non_striker       260920 non-null  object
 9   batsman_runs      260920 non-null  int64 
 10  extra_runs        260920 non-null  int64 
 11  total_runs        260920 non-null  int64 
 12  extras_type       14125 non-null   object
 13  is_wicket         260920 non-null  int64 
 14  player_dismissed  12950 non-null   object
 15  dismissal_kind    12950 non-null   object
 16  fielder           9354 non-null    obj

### 3. PREPROCESSING

In [174]:
# --- MATCHES ---

# drop no result matches
matches = matches.dropna(subset=['winner'])


In [175]:
# fill missing values
matches['city'] = matches['city'].fillna('Unknown')
matches['method'] = matches['method'].fillna('Normal')


In [176]:
matches.info()

<class 'pandas.core.frame.DataFrame'>
Index: 1090 entries, 0 to 1094
Data columns (total 20 columns):
 #   Column           Non-Null Count  Dtype  
---  ------           --------------  -----  
 0   id               1090 non-null   int64  
 1   season           1090 non-null   object 
 2   city             1090 non-null   object 
 3   date             1090 non-null   object 
 4   match_type       1090 non-null   object 
 5   player_of_match  1090 non-null   object 
 6   venue            1090 non-null   object 
 7   team1            1090 non-null   object 
 8   team2            1090 non-null   object 
 9   toss_winner      1090 non-null   object 
 10  toss_decision    1090 non-null   object 
 11  winner           1090 non-null   object 
 12  result           1090 non-null   object 
 13  result_margin    1076 non-null   float64
 14  target_runs      1090 non-null   float64
 15  target_overs     1090 non-null   float64
 16  super_over       1090 non-null   object 
 17  method           10

In [177]:
# date processing
matches['date'] = pd.to_datetime(matches['date'])
matches['match_year'] = matches['date'].dt.year
matches['match_month'] = matches['date'].dt.month

In [178]:
# team normalization
team_map = {
    'Delhi Daredevils':'Delhi Capitals',
    'Kings XI Punjab':'Punjab Kings',
    'Rising Pune Supergiant':'Rising Pune Supergiants',
    'Royal Challengers Bangalore':'Royal Challengers Bengaluru'
}

for col in ['team1','team2','toss_winner','winner']:
    matches[col] = matches[col].replace(team_map)

In [179]:
matches.head()

,id,season,city,date,match_type,player_of_match,venue,team1,team2,toss_winner,...,result,result_margin,target_runs,target_overs,super_over,method,umpire1,umpire2,match_year,match_month
0,335982,2007/08,Bangalore,2008-04-18,League,BB McCullum,M Chinnaswamy Stadium,Royal Challengers Bengaluru,Kolkata Knight Riders,Royal Challengers Bengaluru,...,runs,140.0,223.0,20.0,N,Normal,Asad Rauf,RE Koertzen,2008,4
1,335983,2007/08,Chandigarh,2008-04-19,League,MEK Hussey,"Punjab Cricket Association Stadium, Mohali",Punjab Kings,Chennai Super Kings,Chennai Super Kings,...,runs,33.0,241.0,20.0,N,Normal,MR Benson,SL Shastri,2008,4
2,335984,2007/08,Delhi,2008-04-19,League,MF Maharoof,Feroz Shah Kotla,Delhi Capitals,Rajasthan Royals,Rajasthan Royals,...,wickets,9.0,130.0,20.0,N,Normal,Aleem Dar,GA Pratapkumar,2008,4
3,335985,2007/08,Mumbai,2008-04-20,League,MV Boucher,Wankhede Stadium,Mumbai Indians,Royal Challengers Bengaluru,Mumbai Indians,...,wickets,5.0,166.0,20.0,N,Normal,SJ Davis,DJ Harper,2008,4
4,335986,2007/08,Kolkata,2008-04-20,League,DJ Hussey,Eden Gardens,Kolkata Knight Riders,Deccan Chargers,Deccan Chargers,...,wickets,5.0,111.0,20.0,N,Normal,BF Bowden,K Hariharan,2008,4


In [180]:
# --- DELIVERIES ---

# filter innings
deliveries = deliveries[deliveries['inning'].isin([1,2])]

In [181]:
# fill NA
deliveries['extras_type'] = deliveries['extras_type'].fillna('normal')
deliveries['player_dismissed'] = deliveries['player_dismissed'].fillna('none')
deliveries['dismissal_kind'] = deliveries['dismissal_kind'].fillna('none')
deliveries['fielder'] = deliveries['fielder'].fillna('none')

In [182]:
deliveries.info()

<class 'pandas.core.frame.DataFrame'>
Index: 260759 entries, 0 to 260919
Data columns (total 17 columns):
 #   Column            Non-Null Count   Dtype 
---  ------            --------------   ----- 
 0   match_id          260759 non-null  int64 
 1   inning            260759 non-null  int64 
 2   batting_team      260759 non-null  object
 3   bowling_team      260759 non-null  object
 4   over              260759 non-null  int64 
 5   ball              260759 non-null  int64 
 6   batter            260759 non-null  object
 7   bowler            260759 non-null  object
 8   non_striker       260759 non-null  object
 9   batsman_runs      260759 non-null  int64 
 10  extra_runs        260759 non-null  int64 
 11  total_runs        260759 non-null  int64 
 12  extras_type       260759 non-null  object
 13  is_wicket         260759 non-null  int64 
 14  player_dismissed  260759 non-null  object
 15  dismissal_kind    260759 non-null  object
 16  fielder           260759 non-null  object
d

### 4. FEATURE ENGINEERING

In [183]:
# only first 10 overs
d10 = deliveries[(deliveries['over'] < 10) & (deliveries['inning'] == 1)]

In [184]:
d10

,match_id,inning,batting_team,bowling_team,over,ball,batter,bowler,non_striker,batsman_runs,extra_runs,total_runs,extras_type,is_wicket,player_dismissed,dismissal_kind,fielder
0,335982,1,Kolkata Knight Riders,Royal Challengers Bangalore,0,1,SC Ganguly,P Kumar,BB McCullum,0,1,1,legbyes,0,none,none,none
1,335982,1,Kolkata Knight Riders,Royal Challengers Bangalore,0,2,BB McCullum,P Kumar,SC Ganguly,0,0,0,normal,0,none,none,none
2,335982,1,Kolkata Knight Riders,Royal Challengers Bangalore,0,3,BB McCullum,P Kumar,SC Ganguly,0,1,1,wides,0,none,none,none
3,335982,1,Kolkata Knight Riders,Royal Challengers Bangalore,0,4,BB McCullum,P Kumar,SC Ganguly,0,0,0,normal,0,none,none,none
4,335982,1,Kolkata Knight Riders,Royal Challengers Bangalore,0,5,BB McCullum,P Kumar,SC Ganguly,0,0,0,normal,0,none,none,none
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
260796,1426312,1,Sunrisers Hyderabad,Kolkata Knight Riders,9,2,AK Markram,SP Narine,H Klaasen,1,0,1,normal,0,none,none,none
260797,1426312,1,Sunrisers Hyderabad,Kolkata Knight Riders,9,3,H Klaasen,SP Narine,AK Markram,1,0,1,normal,0,none,none,none
260798,1426312,1,Sunrisers Hyderabad,Kolkata Knight Riders,9,4,AK Markram,SP Narine,H Klaasen,0,0,0,normal,0,none,none,none
260799,1426312,1,Sunrisers Hyderabad,Kolkata Knight Riders,9,5,AK Markram,SP Narine,H Klaasen,0,0,0,normal,0,none,none,none


In [185]:
d10.info()

<class 'pandas.core.frame.DataFrame'>
Index: 67939 entries, 0 to 260800
Data columns (total 17 columns):
 #   Column            Non-Null Count  Dtype 
---  ------            --------------  ----- 
 0   match_id          67939 non-null  int64 
 1   inning            67939 non-null  int64 
 2   batting_team      67939 non-null  object
 3   bowling_team      67939 non-null  object
 4   over              67939 non-null  int64 
 5   ball              67939 non-null  int64 
 6   batter            67939 non-null  object
 7   bowler            67939 non-null  object
 8   non_striker       67939 non-null  object
 9   batsman_runs      67939 non-null  int64 
 10  extra_runs        67939 non-null  int64 
 11  total_runs        67939 non-null  int64 
 12  extras_type       67939 non-null  object
 13  is_wicket         67939 non-null  int64 
 14  player_dismissed  67939 non-null  object
 15  dismissal_kind    67939 non-null  object
 16  fielder           67939 non-null  object
dtypes: int64(8), obj

In [186]:
d10 = d10[['match_id','inning','batting_team','bowling_team',
           'total_runs','is_wicket','batsman_runs','ball']]


In [187]:
d10

,match_id,inning,batting_team,bowling_team,total_runs,is_wicket,batsman_runs,ball
0,335982,1,Kolkata Knight Riders,Royal Challengers Bangalore,1,0,0,1
1,335982,1,Kolkata Knight Riders,Royal Challengers Bangalore,0,0,0,2
2,335982,1,Kolkata Knight Riders,Royal Challengers Bangalore,1,0,0,3
3,335982,1,Kolkata Knight Riders,Royal Challengers Bangalore,0,0,0,4
4,335982,1,Kolkata Knight Riders,Royal Challengers Bangalore,0,0,0,5
...,...,...,...,...,...,...,...,...
260796,1426312,1,Sunrisers Hyderabad,Kolkata Knight Riders,1,0,1,2
260797,1426312,1,Sunrisers Hyderabad,Kolkata Knight Riders,1,0,1,3
260798,1426312,1,Sunrisers Hyderabad,Kolkata Knight Riders,0,0,0,4
260799,1426312,1,Sunrisers Hyderabad,Kolkata Knight Riders,0,0,0,5


In [188]:
agg

,match_id,inning,runs_first10,wickets_first10,boundaries_first10,extras_first10
0,335982,1,87,1,11,10
1,335982,2,51,7,3,14
2,335983,1,88,3,14,3
3,335983,2,110,1,17,4
4,335984,1,57,4,7,5
...,...,...,...,...,...,...
2175,1426310,2,85,2,13,1
2176,1426311,1,99,4,14,1
2177,1426311,2,73,3,9,3
2178,1426312,1,61,4,7,7


In [189]:
# aggregate
agg = d10.groupby(['match_id','inning','batting_team','bowling_team']).agg({
    'total_runs':'sum',
    'is_wicket':'sum',
    'batsman_runs': lambda x: (x==4).sum() + (x==6).sum(),
    'ball':'count'
}).reset_index()
agg.columns = ['match_id','inning','batting_team','bowling_team',
               'runs_10','wickets_10','boundaries_10','balls_10']

In [190]:
agg

,match_id,inning,batting_team,bowling_team,runs_10,wickets_10,boundaries_10,balls_10
0,335982,1,Kolkata Knight Riders,Royal Challengers Bangalore,87,1,11,62
1,335983,1,Chennai Super Kings,Kings XI Punjab,88,3,14,62
2,335984,1,Rajasthan Royals,Delhi Daredevils,57,4,7,61
3,335985,1,Mumbai Indians,Royal Challengers Bangalore,74,3,11,62
4,335986,1,Deccan Chargers,Kolkata Knight Riders,52,4,5,64
...,...,...,...,...,...,...,...,...
1090,1426307,1,Punjab Kings,Sunrisers Hyderabad,99,1,15,60
1091,1426309,1,Sunrisers Hyderabad,Kolkata Knight Riders,92,4,12,65
1092,1426310,1,Royal Challengers Bengaluru,Rajasthan Royals,76,2,9,61
1093,1426311,1,Sunrisers Hyderabad,Rajasthan Royals,99,4,14,60


In [191]:
agg

,match_id,inning,batting_team,bowling_team,runs_10,wickets_10,boundaries_10,balls_10
0,335982,1,Kolkata Knight Riders,Royal Challengers Bangalore,87,1,11,62
1,335983,1,Chennai Super Kings,Kings XI Punjab,88,3,14,62
2,335984,1,Rajasthan Royals,Delhi Daredevils,57,4,7,61
3,335985,1,Mumbai Indians,Royal Challengers Bangalore,74,3,11,62
4,335986,1,Deccan Chargers,Kolkata Knight Riders,52,4,5,64
...,...,...,...,...,...,...,...,...
1090,1426307,1,Punjab Kings,Sunrisers Hyderabad,99,1,15,60
1091,1426309,1,Sunrisers Hyderabad,Kolkata Knight Riders,92,4,12,65
1092,1426310,1,Royal Challengers Bengaluru,Rajasthan Royals,76,2,9,61
1093,1426311,1,Sunrisers Hyderabad,Rajasthan Royals,99,4,14,60


In [192]:
# run rate
agg['run_rate_10'] = agg['runs_10'] / (agg['balls_10']/6)

In [193]:
# merge with matches
df = agg.merge(matches, left_on='match_id', right_on='id')


In [194]:
df.sample(5)

,match_id,inning,batting_team,bowling_team,runs_10,wickets_10,boundaries_10,balls_10,run_rate_10,id,...,result,result_margin,target_runs,target_overs,super_over,method,umpire1,umpire2,match_year,match_month
507,829809,1,Kings XI Punjab,Chennai Super Kings,56,5,8,61,5.508197,829809,...,wickets,7.0,131.0,20.0,N,Normal,CK Nandan,C Shamshuddin,2015,5
203,501227,1,Delhi Daredevils,Royal Challengers Bangalore,69,3,8,61,6.786885,501227,...,wickets,3.0,161.0,20.0,N,Normal,S Asnani,RJ Tucker,2011,4
332,598009,1,Rajasthan Royals,Pune Warriors,72,1,10,61,7.081967,598009,...,wickets,7.0,146.0,20.0,N,Normal,M Erasmus,K Srinath,2013,4
426,733989,1,Sunrisers Hyderabad,Rajasthan Royals,74,3,10,61,7.278689,733989,...,runs,32.0,135.0,20.0,N,Normal,AK Chaudhary,NJ Llong,2014,5
580,1082597,1,Kolkata Knight Riders,Mumbai Indians,79,3,8,66,7.181818,1082597,...,wickets,4.0,179.0,20.0,N,Normal,Nitin Menon,CK Nandan,2017,4


In [195]:
final_score = deliveries[deliveries['inning'] == 1] \
                .groupby(['match_id','inning'])['total_runs'] \
                .sum().reset_index()

In [196]:
final_score

,match_id,inning,total_runs
0,335982,1,222
1,335983,1,240
2,335984,1,129
3,335985,1,165
4,335986,1,110
...,...,...,...
1090,1426307,1,214
1091,1426309,1,159
1092,1426310,1,172
1093,1426311,1,175


In [197]:
df = df.merge(final_score, on=['match_id','inning'])
df.rename(columns={'total_runs':'final_score'}, inplace=True)

In [198]:
df

,match_id,inning,batting_team,bowling_team,runs_10,wickets_10,boundaries_10,balls_10,run_rate_10,id,...,result_margin,target_runs,target_overs,super_over,method,umpire1,umpire2,match_year,match_month,final_score
0,335982,1,Kolkata Knight Riders,Royal Challengers Bangalore,87,1,11,62,8.419355,335982,...,140.0,223.0,20.0,N,Normal,Asad Rauf,RE Koertzen,2008,4,222
1,335983,1,Chennai Super Kings,Kings XI Punjab,88,3,14,62,8.516129,335983,...,33.0,241.0,20.0,N,Normal,MR Benson,SL Shastri,2008,4,240
2,335984,1,Rajasthan Royals,Delhi Daredevils,57,4,7,61,5.606557,335984,...,9.0,130.0,20.0,N,Normal,Aleem Dar,GA Pratapkumar,2008,4,129
3,335985,1,Mumbai Indians,Royal Challengers Bangalore,74,3,11,62,7.161290,335985,...,5.0,166.0,20.0,N,Normal,SJ Davis,DJ Harper,2008,4,165
4,335986,1,Deccan Chargers,Kolkata Knight Riders,52,4,5,64,4.875000,335986,...,5.0,111.0,20.0,N,Normal,BF Bowden,K Hariharan,2008,4,110
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1085,1426307,1,Punjab Kings,Sunrisers Hyderabad,99,1,15,60,9.900000,1426307,...,4.0,215.0,20.0,N,Normal,Nitin Menon,VK Sharma,2024,5,214
1086,1426309,1,Sunrisers Hyderabad,Kolkata Knight Riders,92,4,12,65,8.492308,1426309,...,8.0,160.0,20.0,N,Normal,AK Chaudhary,R Pandit,2024,5,159
1087,1426310,1,Royal Challengers Bengaluru,Rajasthan Royals,76,2,9,61,7.475410,1426310,...,4.0,173.0,20.0,N,Normal,KN Ananthapadmanabhan,MV Saidharshan Kumar,2024,5,172
1088,1426311,1,Sunrisers Hyderabad,Rajasthan Royals,99,4,14,60,9.900000,1426311,...,36.0,176.0,20.0,N,Normal,Nitin Menon,VK Sharma,2024,5,175


### 5. ENCODING + IMPUTATION

In [199]:

import joblib



cat_cols = ['batting_team','bowling_team','city','venue','toss_decision']

encoders = {}

for col in cat_cols:
    le = LabelEncoder()
    df[col] = le.fit_transform(df[col].astype(str))
    encoders[col] = le

# SAVE encoders file
joblib.dump(encoders, "encoders.pkl")

['encoders.pkl']

In [200]:
df.sample(5)

,match_id,inning,batting_team,bowling_team,runs_10,wickets_10,boundaries_10,balls_10,run_rate_10,id,...,result_margin,target_runs,target_overs,super_over,method,umpire1,umpire2,match_year,match_month,final_score
143,419134,1,3,13,79,4,10,62,7.645161,419134,...,67.0,189.0,20.0,N,Normal,HDPK Dharmasena,SJA Taufel,2010,3,188
45,336027,1,8,13,59,2,10,62,5.709677,336027,...,6.0,148.0,20.0,N,Normal,BG Jerling,RE Koertzen,2008,5,147
920,1304095,1,16,0,79,3,9,63,7.523810,1304095,...,13.0,174.0,20.0,N,Normal,KN Ananthapadmanabhan,MA Gough,2022,5,173
469,829729,1,10,0,62,4,9,62,6.000000,829729,...,6.0,184.0,20.0,N,Normal,AK Chaudhary,M Erasmus,2015,4,183
1023,1422123,1,5,10,82,2,10,62,7.935484,1422123,...,6.0,169.0,20.0,N,Normal,VA Kulkarni,VK Sharma,2024,3,168


### TRAIN TEST SPLIT

In [201]:
X = df.drop(['final_score','match_id','id'], axis=1)
y = df['final_score']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

print("Successfully preprocessed data and split into train/test sets")

ValueError: could not convert string to float: '2020/21'

### MODEL TRAINING

In [ ]:
lr = LinearRegression()
lr.fit(X_train_scaled, y_train)
pred_lr = lr.predict(X_test_scaled)
print("Linear Regression predictions completed.")

Linear Regression predictions completed.


In [ ]:
dt = DecisionTreeRegressor(max_depth=10, random_state=42)
dt.fit(X_train, y_train)
pred_dt = dt.predict(X_test)
print("Decision Tree predictions completed.")

Decision Tree predictions completed.


In [ ]:
rf = RandomForestRegressor(n_estimators=200, max_depth=15, random_state=42, n_jobs=-1)
rf.fit(X_train, y_train)
pred_rf = rf.predict(X_test)
print("Random Forest predictions completed.")

Random Forest predictions completed.


In [ ]:
svr = SVR(C=200, gamma=0.05)
svr.fit(X_train_scaled, y_train)
pred_svr = svr.predict(X_test_scaled)
print("SVR predictions completed.")

SVR predictions completed.


### EVALUATION TABLE

In [ ]:
def evaluate(y_true, y_pred):
    mse = mean_squared_error(y_true, y_pred)
    rmse = np.sqrt(mse)
    r2 = r2_score(y_true, y_pred)
    return mse, rmse, r2

results = pd.DataFrame({
    'Model':['Linear','Decision Tree','Random Forest','SVR'],
    'MSE':[evaluate(y_test,pred_lr)[0],
           evaluate(y_test,pred_dt)[0],
           evaluate(y_test,pred_rf)[0],
           evaluate(y_test,pred_svr)[0]],
    'RMSE':[evaluate(y_test,pred_lr)[1],
            evaluate(y_test,pred_dt)[1],
            evaluate(y_test,pred_rf)[1],
            evaluate(y_test,pred_svr)[1]],
    'R2':[evaluate(y_test,pred_lr)[2],
          evaluate(y_test,pred_dt)[2],
          evaluate(y_test,pred_rf)[2],
          evaluate(y_test,pred_svr)[2]]
})

print(results)

           Model        MSE      RMSE        R2
0         Linear  33.796143  5.813445  0.966932
1  Decision Tree  23.440367  4.841525  0.977065
2  Random Forest  27.714320  5.264439  0.972883
3            SVR  35.771572  5.980934  0.964999


###  STACKING

In [ ]:
stack = StackingRegressor(
    estimators=[
        ('lr', LinearRegression()),
        ('rf', RandomForestRegressor(n_estimators=200, max_depth=15))
    ],
    final_estimator=LinearRegression(),
    cv=5
)

stack.fit(X_train, y_train)
pred_stack = stack.predict(X_test)

print("STACKING:", evaluate(y_test, pred_stack))

STACKING: (26.282008200916604, np.float64(5.126598111898045), 0.9742844254333367)


### BOOSTING MODELS

In [ ]:
ada = AdaBoostRegressor(n_estimators=100, random_state=42)
ada.fit(X_train, y_train)
pred_ada = ada.predict(X_test)
print("AdaBoost predictions completed.")

AdaBoost predictions completed.


In [ ]:
gb = GradientBoostingRegressor()
gb.fit(X_train, y_train)
pred_gb = gb.predict(X_test)
print("Gradient Boosting predictions completed.")

Gradient Boosting predictions completed.


In [ ]:
xgb = XGBRegressor(n_estimators=200, max_depth=6)
xgb.fit(X_train, y_train)
pred_xgb = xgb.predict(X_test)
print("XGBoost predictions completed.")

XGBoost predictions completed.


In [ ]:
lgbm = LGBMRegressor(n_estimators=200)
lgbm.fit(X_train, y_train)
pred_lgbm = lgbm.predict(X_test)
print("LightGBM predictions completed.")

[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.000301 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 703
[LightGBM] [Info] Number of data points in the train set: 872, number of used features: 14
[LightGBM] [Info] Start training from score 167.008028
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
LightGBM predictions completed.


In [ ]:
boost_results = pd.DataFrame({
    'Model':['AdaBoost','GradientBoost','XGBoost','LightGBM'],
    'MSE':[evaluate(y_test,pred_ada)[0],
           evaluate(y_test,pred_gb)[0],
           evaluate(y_test,pred_xgb)[0],
           evaluate(y_test,pred_lgbm)[0]],
    'RMSE':[evaluate(y_test,pred_ada)[1],
            evaluate(y_test,pred_gb)[1],
            evaluate(y_test,pred_xgb)[1],
            evaluate(y_test,pred_lgbm)[1]],
    'R2':[evaluate(y_test,pred_ada)[2],
          evaluate(y_test,pred_gb)[2],
          evaluate(y_test,pred_xgb)[2],
          evaluate(y_test,pred_lgbm)[2]]
})

print(boost_results)

           Model        MSE      RMSE        R2
0       AdaBoost  56.444698  7.512969  0.944772
1  GradientBoost  33.023452  5.746604  0.967688
2        XGBoost  21.937761  4.683776  0.978535
3       LightGBM  42.922539  6.551530  0.958003


In [ ]:
import joblib

joblib.dump(rf, "model.pkl")          # or stack
joblib.dump(scaler, "scaler.pkl")
joblib.dump(le, "encoder.pkl")

['encoder.pkl']

In [ ]:
import numpy as np
import pandas as pd
import joblib

# load
model = joblib.load("model.pkl")
scaler = joblib.load("scaler.pkl")
encoders = joblib.load("encoders.pkl")   # now this exists

def predict_score():
    
    print("---- ENTER MATCH DETAILS ----")
    
    batting_team = 'Mumbai Indians'
    bowling_team = 'Chennai Super Kings'
    city = 'Mumbai'
    venue = 'Wankhede Stadium'
    toss_decision = 'bat'
    
    runs_10 = 85
    wickets_10 = 2 
    boundaries_10 = 10  
    balls_10 = 60
    
    # derived
    run_rate_10 = runs_10 / (balls_10/6)
    
    # dataframe
    input_df = pd.DataFrame([{
        'batting_team': batting_team,
        'bowling_team': bowling_team,
        'city': city,
        'venue': venue,
        'toss_decision': toss_decision,
        'runs_10': runs_10,
        'wickets_10': wickets_10,
        'boundaries_10': boundaries_10,
        'balls_10': balls_10,
        'run_rate_10': run_rate_10
    }])
    
    # encoding (correct way)
    for col in ['batting_team','bowling_team','city','venue','toss_decision']:
        input_df[col] = encoders[col].transform(input_df[col])
    
    # scaling
    input_scaled = scaler.transform(input_df)
    
    # prediction
    pred = model.predict(input_scaled)[0]
    
    print(f"\n🏏 Predicted First Innings Score: {int(pred)} runs")

# run
predict_score()

FileNotFoundError: [Errno 2] No such file or directory: 'encoders.pkl'

In [ ]:
# predict_score()

---- ENTER MATCH DETAILS ----


FileNotFoundError: [Errno 2] No such file or directory: 'encoders.pkl'